# InfernoTactics v10: Fire-Relative Multi-Dispatch RL

This notebook runs the v10 relative-action training and evaluation pipeline:

  - Fire-relative semantic action space
  - Policy-decided list-only multi-dispatch per simulation tick
  - Synthetic deterministic traffic model with BPR-style congestion
  - Configurable per-resource response delays

It calls the implementation in `src/train/train_relative.py`; it does not duplicate the training loop.

## 1. Configure the Environment

Set `PROJECT_ROOT` to the folder containing `src/`, `data/`, and `models/`. This is typically:

```
/content/drive/MyDrive/InfernoTactics/infernotactics
```

or a local path under `cosmos-venv`.

In [ ]:
import glob
import os
import sys
from pathlib import Path

# Change this to the folder containing src/, data/, and models/.
PROJECT_ROOT = "/content/drive/MyDrive/InfernoTactics/infernotactics"

# Auto-detect if the project is already cloned under /content.
if not os.path.exists(os.path.join(PROJECT_ROOT, "src", "train", "train_relative.py")):
    matches = glob.glob("/content/**/src/train/train_relative.py", recursive=True)
    if matches:
        PROJECT_ROOT = str(Path(matches[0]).parents[2])

PROJECT_ROOT = os.path.abspath(PROJECT_ROOT)
required = [
    os.path.join(PROJECT_ROOT, "src", "train", "train_relative.py"),
    os.path.join(PROJECT_ROOT, "data", "grid_static.npy"),
]
if not all(os.path.exists(p) for p in required):
    raise RuntimeError("PROJECT_ROOT must point to the project folder containing src/ and data/.")

os.environ["PYTHONPATH"] = PROJECT_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Install Dependencies

Install the project dependencies in the connected Colab kernel.

In [ ]:
%pip install -q -r {os.path.join(PROJECT_ROOT, "requirements.txt")}

## 3. Verify the Fire-Relative Action Boundary

The same semantic action must resolve to different absolute zones when the ignition moves. This is the core invariant missing from the original absolute-zone policy.

In [ ]:
from src.env.inferno_env import InfernoEnv
from src.train.relative_actions import TARGET_TYPES, resolve_relative_targets

env = InfernoEnv(seed=8200)
obs_a = env.reset(ignition_point=(207, 222), seed=1)
zones_a, _ = resolve_relative_targets(env, obs_a)
obs_b = env.reset(ignition_point=(57, 371), seed=1)
zones_b, _ = resolve_relative_targets(env, obs_b)
active = TARGET_TYPES.index("active_fire")
print("active_fire zone at Skull Rock:", zones_a[0, active])
print("active_fire zone at Mandeville:", zones_b[0, active])
assert zones_a[0, active] != zones_b[0, active]
print("Relative-action invariant passed.")

## 4. Smoke Test

Run a short training job first to confirm the pipeline executes end-to-end. The smoke test uses a distinct run tag so its outputs do not overwrite the main run.

In [ ]:
import subprocess

run_env = os.environ.copy()
run_env.update({
    "INFERNO_N_EPISODES": "10",
    "INFERNO_V8_EVAL_EVERY": "5",
    "INFERNO_RUN_TAG": "relative_v10_smoke",
    "INFERNO_V8_CHECKPOINT_EVERY": "2",
    "INFERNO_TRACE_EVERY": "5",
    "PYTHONPATH": PROJECT_ROOT + os.pathsep + run_env.get("PYTHONPATH", ""),
})
subprocess.run([sys.executable, "-m", "src.train.train_relative"], cwd=PROJECT_ROOT, env=run_env, check=True)

## 5. Full Training

Run this after the smoke test succeeds. The default is 100 randomized-ignition episodes. Checkpoints are written under the project folder in `models/checkpoints_<run_tag>/`.

In [ ]:
N_EPISODES = 100
RUN_TAG = "relative_v10_multi_dispatch_100"

run_env = os.environ.copy()
run_env.update({
    "INFERNO_N_EPISODES": str(N_EPISODES),
    "INFERNO_V8_EVAL_EVERY": "20",
    "INFERNO_RUN_TAG": RUN_TAG,
    "INFERNO_V8_CHECKPOINT_EVERY": "20",
    "INFERNO_TRACE_EVERY": "20",
    "PYTHONPATH": PROJECT_ROOT + os.pathsep + run_env.get("PYTHONPATH", ""),
})
subprocess.run([sys.executable, "-m", "src.train.train_relative"], cwd=PROJECT_ROOT, env=run_env, check=True)

In [ ]:
from pathlib import Path
checkpoint_dir = Path(PROJECT_ROOT) / "models" / f"checkpoints_{RUN_TAG}"
print("Checkpoint directory:", checkpoint_dir)
print("Checkpoints:", sorted(p.name for p in checkpoint_dir.glob("*.pt"))[-10:])

## 6. Evaluate the Final Checkpoint

Run a 30-point randomized evaluation against the latest checkpoint.

In [ ]:
subprocess.run([
    sys.executable,
    "-m",
    "src.train.eval_relative",
    "--checkpoint",
    str(checkpoint_dir / "latest.pt"),
    "--random-points",
    "30",
    "--episodes",
    "1",
], cwd=PROJECT_ROOT, check=True)

## 7. Generate the Training Dashboard

Produce a matplotlib dashboard from the per-run CSV logs.

In [ ]:
subprocess.run([
    sys.executable,
    "-m",
    "src.train.plot_training",
    "--run-tag",
    RUN_TAG,
], cwd=PROJECT_ROOT, check=True)

## Interpretation

Compare v10 against historical checkpoints. The important diagnostics are:

- held-out reward and containment improve, not only the Skull Rock anchor
- the selected semantic target changes as the fire moves
- `active_fire` resolves to different absolute zones for different ignitions
- the policy does not lock onto one absolute `(resource, zone)` pair
- resource-specific target behavior is sensible: suppression targets active fire, trench targets adjacent fuel, rescue targets threatened population

For embedding this policy in another simulation, see `integration-instructions.txt`.